In [1]:
from langchain_groq import ChatGroq

In [2]:
llm = ChatGroq(
    temperature=0, 
    groq_api_key='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', 
    model_name="llama-3.3-70b-versatile"
)
response = llm.invoke("The first person to land on moon was ...")
print(response.content)

The first person to land on the moon was Neil Armstrong. He stepped out of the lunar module Eagle and onto the moon's surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That's one small step for man, one giant leap for mankind," as he became the first human to set foot on the moon.


In [5]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://jobs.nike.com/job/R-33460")
page_data = loader.load().pop().page_content
print(page_data)

USER_AGENT environment variable not set, consider setting it to identify your requests.






















Nike Careers









































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu



Return to Previous Menu



Select a Langua

In [6]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):    
        """
)

chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)

str

In [7]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

[{'role': 'Retail Associate, PT - Nike Frontenac',
  'experience': 'Part Time',
  'skills': '',
  'description': 'Retail Associate, PT - Nike Frontenac (14-29 hours/week)'},
 {'role': 'Kaufmann/-frau im Einzelhandel (Ausbildung) - 37.5H - Metzingen',
  'experience': '',
  'skills': '',
  'description': 'Kaufmann/-frau im Einzelhandel (Ausbildung) - 37.5H - Metzingen (w/m/d)'},
 {'role': 'Senior Product Manager - Transportation Management System (TMS), ITC',
  'experience': 'Senior',
  'skills': '',
  'description': 'Senior Product Manager - Transportation Management System (TMS), ITC'},
 {'role': 'Retail Associate, PT - Nike Oklahoma City',
  'experience': 'Part Time',
  'skills': '',
  'description': 'Retail Associate, PT - Nike Oklahoma City (14-29 hours/week)'},
 {'role': 'Software Engineer II, ITC',
  'experience': '',
  'skills': '',
  'description': 'Software Engineer II, ITC'},
 {'role': 'Senior Professional – Producer, Brand Creative EMEA, Production',
  'experience': 'Senior',

In [8]:
type(json_res)

list

In [10]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [11]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [16]:
links = collection.query(query_texts=job['skills'], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/android-tv-portfolio'},
  {'links': 'https://example.com/kotlin-backend-portfolio'}]]

In [17]:
job

{'role': 'Retail Associate, PT - Nike Frontenac',
 'experience': 'Part Time',
 'skills': '',
 'description': 'Retail Associate, PT - Nike Frontenac (14-29 hours/week)'}

In [18]:
job = json_res
job['skills']

''

In [15]:
# If json_res is a list, take the first element
if isinstance(json_res, list):
    json_res = json_res[0]

job = json_res
job['skills']


''

In [19]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are Arva Sukeerthan, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ 
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase Atliq's portfolio: {link_list}
        Remember you are Mohan, BDE at AtliQ. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        
        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Enhance Your Retail Operations with AtliQ's Automated Solutions

Dear Hiring Manager,

I came across the job posting for a Retail Associate at Nike Frontenac, and I understand the importance of streamlining retail operations to provide an exceptional customer experience. As a Business Development Executive at AtliQ, I'd like to introduce you to our AI and software consulting services that can help optimize your retail processes.

At AtliQ, we specialize in developing tailored solutions that cater to the unique needs of businesses like yours. Our expertise in automation can help you reduce costs, enhance efficiency, and scale your operations. We've empowered numerous enterprises with our customized solutions, and I'd like to highlight a few examples from our portfolio:

* Our work on Android TV solutions (https://example.com/android-tv-portfolio) demonstrates our capability in developing innovative and user-friendly interfaces that can be applied to retail environments.
* Our K